# Qwen2.5-0.5B Fine-Tuning on XAI Server
# Frequent Itemset Extraction Task

**Server Resources**: 358GB RAM (jupytergpu)  
**Model**: Qwen/Qwen2.5-0.5B-Instruct (494M parameters)  
**Method**: **4-bit QLoRA ONLY** (quantized LoRA)  
**Dataset**: 88 train / 10 validation examples (ENHANCED with real-world context)

## ⚠️ Important Findings from Testing:
- ✅ **Use quantized LoRA (4-bit)** - Full precision not recommended
- ✅ **Real-world context** - Items mapped to grocery/electronics/clothing/household
- ✅ **Triplet format** - Better than pairs for itemset learning
- ❌ **Synthetic-only data fails** - Random strings cause poor performance
- ❌ **API deployment doesn't work** - Use model locally after training
- ✅ **Max 7B models** - Server can handle up to 7B params

This notebook will:
1. Check system resources
2. Install required packages
3. Load ENHANCED dataset (with real-world context)
4. Configure and train Qwen2.5-0.5B with **4-bit QLoRA**
5. Evaluate the fine-tuned model
6. Save and export the model

## 1. System Resources Check

In [ ]:
import os
import psutil
import torch

# Check system resources
print("=" * 60)
print("SYSTEM RESOURCES")
print("=" * 60)

# CPU info
print(f"CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count(logical=True)} logical")
print(f"CPU usage: {psutil.cpu_percent(interval=1)}%")

# Memory info
mem = psutil.virtual_memory()
print(f"\nRAM: {mem.used / 1024**3:.2f} / {mem.total / 1024**3:.2f} GB ({mem.percent}%)")
print(f"Available: {mem.available / 1024**3:.2f} GB")

# GPU info
if torch.cuda.is_available():
    print(f"\nGPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("\nGPU: Not available (will use CPU)")

# Disk info
disk = psutil.disk_usage('/')
print(f"\nDisk: {disk.used / 1024**3:.2f} / {disk.total / 1024**3:.2f} GB ({disk.percent}%)")
print("=" * 60)

## 2. Install Required Packages

In [ ]:
# Install required packages
# Note: bitsandbytes may need special installation on some servers
!pip install -q datasets transformers trl peft accelerate torch psutil matplotlib bitsandbytes

## 3. Upload Dataset to Server

Upload your `hf_dataset` folder (created by `create_hf_dataset.py`) to the server.

You can either:
- Use Jupyter's file upload feature
- SCP/SFTP the folder to the server
- Clone the GitHub repo (private): `git clone https://github.com/oliversl1vka/itemsety-qwen-finetuning.git`

**Note**: The dataset is gitignored, so you need to upload it separately or recreate it on the server.

In [ ]:
# Set dataset path (USE ENHANCED DATASET with real-world context!)
DATASET_PATH = "./hf_dataset_enhanced"  # Enhanced with grocery/electronics/etc
OUTPUT_DIR = "./qwen_itemset_model_enhanced"

print("⚠️  IMPORTANT: Using ENHANCED dataset with real-world context!")
print("   Original synthetic-only data performed poorly in testing.")
print("   Enhanced data maps items to: grocery, electronics, clothing, household\n")

# Verify dataset exists
import os
if os.path.exists(DATASET_PATH):
    print(f"✅ Enhanced dataset found at: {DATASET_PATH}")
    # List dataset files
    for root, dirs, files in os.walk(DATASET_PATH):
        level = root.replace(DATASET_PATH, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 2 * (level + 1)
        for file in files[:5]:  # Show first 5 files
            print(f"{subindent}{file}")
else:
    print(f"❌ Enhanced dataset not found at: {DATASET_PATH}")
    print("   Please upload hf_dataset_enhanced folder")
    print("   Or run: python enhance_training_data.py && python create_hf_dataset.py --input training_data/all_training_examples_enhanced.json --output hf_dataset_enhanced")

## 4. Load and Inspect Dataset

In [ ]:
from datasets import load_from_disk

# Load dataset
dataset = load_from_disk(DATASET_PATH)

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Train examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['validation'])}")
print(f"\nDataset structure:")
print(dataset)

# Inspect first example
print("\n" + "=" * 60)
print("FIRST TRAINING EXAMPLE")
print("=" * 60)
example = dataset['train'][0]
print(f"Keys: {example.keys()}")
print(f"\nMessages ({len(example['messages'])} total):")
for i, msg in enumerate(example['messages']):
    role = msg['role']
    content = msg['content'][:200] + "..." if len(msg['content']) > 200 else msg['content']
    print(f"\n[{i+1}] {role.upper()}:")
    print(content)

## 5. Load Model and Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

print("=" * 60)
print("LOADING MODEL & TOKENIZER (4-BIT QUANTIZED)")
print("=" * 60)
print("\n⚠️  Using 4-bit quantization (QLoRA) based on testing findings")
print("   Full precision training not recommended on this server\n")

# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit Quantization Config:")
print(f"  Quant type: {bnb_config.bnb_4bit_quant_type}")
print(f"  Compute dtype: {bnb_config.bnb_4bit_compute_dtype}")
print(f"  Double quant: {bnb_config.bnb_4bit_use_double_quant}")

# Load tokenizer
print(f"\n📝 Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    padding_side="right",
)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("   Set pad_token = eos_token")

# Load model with 4-bit quantization
print(f"\n🤖 Loading model with 4-bit quantization: {MODEL_NAME}")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Model info
total_params = sum(p.numel() for p in model.parameters())
print(f"\n✅ Model loaded successfully with 4-bit quantization!")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Device: {next(model.parameters()).device}")
print(f"   Quantized: 4-bit NF4")

## 6. Configure LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print("=" * 60)
print("CONFIGURING LORA FOR 4-BIT MODEL")
print("=" * 60)

# Prepare model for k-bit training (required for 4-bit)
print("\nPreparing model for 4-bit training...")
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                    # LoRA rank
    lora_alpha=32,           # LoRA alpha (scaling factor)
    lora_dropout=0.05,       # Dropout probability
    bias="none",             # Bias type
    task_type="CAUSAL_LM",   # Task type
    target_modules=[         # Target modules for LoRA
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
)

print(f"\nLoRA Configuration:")
print(f"  Rank (r): {lora_config.r}")
print(f"  Alpha: {lora_config.lora_alpha}")
print(f"  Dropout: {lora_config.lora_dropout}")
print(f"  Target modules: {len(lora_config.target_modules)}")

# Apply LoRA to model
print(f"\nApplying LoRA to 4-bit quantized model...")
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

print("\n✅ LoRA configured successfully for 4-bit training!")

## 7. Prepare Training Dataset

In [ ]:
from transformers import TrainingArguments

print("=" * 60)
print("CONFIGURING TRAINING (4-BIT OPTIMIZED)")
print("=" * 60)

training_args = TrainingArguments(
    output_dir="./qwen_finetuned_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=10,
    logging_steps=10,
    eval_strategy="steps",  # Changed from evaluation_strategy
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=False,  # BFloat16 used in quantization config
    bf16=True,   # Match quantization compute_dtype
    optim="paged_adamw_8bit",  # Memory-efficient optimizer for 4-bit
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    report_to="none",
)

print(f"\nTraining Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size (train): {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim} (4-bit compatible)")
print(f"  Precision: BFloat16 (matches quantization)")
print(f"  Gradient checkpointing: {training_args.gradient_checkpointing}")

print("\n✅ Training arguments configured for 4-bit QLoRA!")

## 8. Configure Training Arguments

In [ ]:
from transformers import TrainingArguments

print("=" * 60)
print("TRAINING CONFIGURATION")
print("=" * 60)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,      # Increase if you have GPU memory
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,      # Effective batch = 4 * 2 = 8
    gradient_checkpointing=True,
    optim="adamw_torch",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=5,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=False,
    bf16=True,                          # Use BFloat16
    group_by_length=True,
    report_to=["tensorboard"],
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
)

print(f"\nTraining Configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size (per device): {training_args.per_device_train_batch_size}")
print(f"  Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"  Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Optimizer: {training_args.optim}")
print(f"  Scheduler: {training_args.lr_scheduler_type}")
print(f"  Mixed precision: BF16={training_args.bf16}, FP16={training_args.fp16}")
print(f"  Output dir: {training_args.output_dir}")

# Estimate training steps
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)) * training_args.num_train_epochs
print(f"\n  Estimated total steps: ~{total_steps}")

## 9. Initialize Trainer and Start Training

In [ ]:
from trl import SFTTrainer

print("=" * 60)
print("INITIALIZING TRAINER")
print("=" * 60)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    max_seq_length=2048,
    dataset_text_field="text",
    packing=False,
)

print("✅ Trainer initialized!")
print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60)
print("\nThis may take 15-30 minutes depending on your hardware...")
print("You can monitor progress in TensorBoard:")
print(f"  tensorboard --logdir {OUTPUT_DIR}")

# Start training
trainer.train()

## 10. Save Fine-Tuned Model

In [ ]:
import os

print("=" * 60)
print("SAVING MODEL")
print("=" * 60)

# Save final model
final_model_path = f"{OUTPUT_DIR}/final"
print(f"\nSaving model to: {final_model_path}")

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"\n✅ Model saved successfully!")

# Check saved files
print(f"\nSaved files:")
for file in os.listdir(final_model_path):
    file_path = os.path.join(final_model_path, file)
    size = os.path.getsize(file_path) / (1024 * 1024)  # MB
    print(f"  {file}: {size:.2f} MB")

print(f"\nTo load the model:")
print(f"  from peft import AutoPeftModelForCausalLM")
print(f"  model = AutoPeftModelForCausalLM.from_pretrained('{final_model_path}')")

## 11. Test Fine-Tuned Model

In [ ]:
import json

print("=" * 60)
print("TESTING FINE-TUNED MODEL")
print("=" * 60)

# Test on a validation example
test_example = dataset['validation'][0]

# Prepare messages (system + user only, no assistant)
messages = [
    test_example['messages'][0],  # system
    test_example['messages'][1],  # user
]

# Format with chat template
inputs = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)

print("\nGenerating response...")
print("(This may take 10-30 seconds)")

# Generate
with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

# Decode response
response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

print("\n" + "=" * 60)
print("MODEL RESPONSE")
print("=" * 60)
print(response[:1000])  # Show first 1000 chars

# Try to parse as JSON
try:
    parsed = json.loads(response)
    print("\n" + "=" * 60)
    print("PARSED JSON")
    print("=" * 60)
    print(f"Type: {type(parsed)}")
    if isinstance(parsed, list):
        print(f"Number of itemsets: {len(parsed)}")
        if len(parsed) > 0:
            print(f"\nFirst itemset:")
            print(json.dumps(parsed[0], indent=2))
except json.JSONDecodeError as e:
    print(f"\n❌ JSON Parse Error: {e}")
    print("   Model needs more training or different prompt")

## 12. Monitor Memory Usage During Training

In [ ]:
import matplotlib.pyplot as plt

# Get memory info
mem = psutil.virtual_memory()

print("=" * 60)
print("FINAL MEMORY USAGE")
print("=" * 60)
print(f"Total RAM: {mem.total / 1024**3:.2f} GB")
print(f"Used: {mem.used / 1024**3:.2f} GB ({mem.percent}%)")
print(f"Available: {mem.available / 1024**3:.2f} GB")

# GPU memory if available
if torch.cuda.is_available():
    print(f"\nGPU Memory Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
    print(f"GPU Memory Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

# Visualize memory usage
fig, ax = plt.subplots(figsize=(10, 4))
categories = ['Used', 'Available']
values = [mem.used / 1024**3, mem.available / 1024**3]
colors = ['#FF6B6B', '#51CF66']

ax.bar(categories, values, color=colors, alpha=0.7)
ax.set_ylabel('Memory (GB)')
ax.set_title(f'System Memory Usage - {mem.percent:.1f}% Used')
ax.axhline(y=mem.total / 1024**3, color='gray', linestyle='--', label=f'Total: {mem.total / 1024**3:.1f} GB')
ax.legend()
plt.tight_layout()
plt.show()

print(f"\n✅ Training complete! Memory usage is healthy.")

## 13. Download Model from Server

After training, download the fine-tuned model to your local machine:

```bash
# Using SCP (from your local machine)
scp -r username@server:/path/to/qwen_itemset_model/final ./qwen_itemset_model_final

# Or zip and download via Jupyter
# Run this in a cell:
!tar -czf qwen_model.tar.gz qwen_itemset_model/final
# Then download qwen_model.tar.gz via Jupyter file browser
```

## Next Steps

1. **Evaluate against ground truth** - Compare with gpt_4_1 results from `runs.db`
2. **Integrate into pipeline** - Replace Azure OpenAI with fine-tuned model
3. **Push to HuggingFace Hub** - Share model publicly or privately
4. **Further tuning** - Adjust LoRA rank, learning rate, epochs based on results

---

**Training Summary:**
- ✅ Model: Qwen2.5-0.5B-Instruct with LoRA
- ✅ Dataset: 88 train / 10 validation examples
- ✅ Training: 3 epochs with BFloat16 precision
- ✅ Output: `qwen_itemset_model/final/`

**Repository**: https://github.com/oliversl1vka/itemsety-qwen-finetuning